In [ ]:
!pip install -q kaggle

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR']= '/content'

In [ ]:
!kaggle datasets download -d disisbig/telugu-wikipedia-articles

100% 199M/199M [00:01<00:00, 180MB/s]
100% 199M/199M [00:01<00:00, 179MB/s]


In [ ]:
!unzip /content/data.zip -d dataset

Archive:  /content/data.zip
   creating: dataset/data/train/
  inflating: dataset/data/train/0.txt  
  inflating: dataset/data/train/1.txt  
  inflating: dataset/data/train/10.txt  
 extracting: dataset/data/train/101.txt  
  inflating: dataset/data/train/102.txt  
  inflating: dataset/data/train/104.txt  
  inflating: dataset/data/train/105.txt  
  inflating: dataset/data/train/106.txt  
  inflating: dataset/data/train/107.txt  
  inflating: dataset/data/train/108.txt  
  inflating: dataset/data/train/109.txt  
  inflating: dataset/data/train/11.txt  
  inflating: dataset/data/train/110.txt  
  inflating: dataset/data/train/111.txt  
  inflating: dataset/data/train/112.txt  
  inflating: dataset/data/train/113.txt  
  inflating: dataset/data/train/114.txt  
  inflating: dataset/data/train/116.txt  
  inflating: dataset/data/train/12.txt  
  inflating: dataset/data/train/13.txt  
  inflating: dataset/data/train/14.txt  
  inflating: dataset/data/train/15.txt  
  inflating: dataset/data

In [2]:
!pip uninstall -y rouge

In [3]:
!pip install rouge

In [5]:
from rouge import Rouge

In [6]:
# Import necessary libraries
import os
import nltk
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# Download NLTK data if not already downloaded
nltk.download('punkt')

# Load Telugu BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertModel.from_pretrained("bert-base-multilingual-cased")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# Tokenize and encode Telugu text
def tokenize_and_encode(text):
    tokens = tokenizer.encode(text, add_special_tokens=True, max_length=512)
    return tokens

# Generate sentence embeddings using Telugu BERT
def generate_sentence_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        token_ids = tokenize_and_encode(sentence)
        with torch.no_grad():
            outputs = model(torch.tensor(token_ids).unsqueeze(0))
        embeddings.append(torch.mean(outputs.last_hidden_state, dim=1).squeeze().numpy())
    return embeddings

In [9]:
# Calculate cosine similarity between sentence embeddings
def calculate_similarity_matrix(embeddings):
    similarity_matrix = cosine_similarity(embeddings)
    return similarity_matrix

In [10]:
# Generate extractive summary from Telugu text
def generate_extractive_summary(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)

    scores = np.zeros(len(sentences))
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    top_sentences_indices = sorted(top_sentences_indices)
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary


In [11]:
# Path to the directory containing Telugu text files
train_data_path = "/content/dataset/data/train"

In [12]:
# Function to read Telugu text files from directory
def read_telugu_text_files(directory):
    texts = []
    for filename in os.listdir(directory):
        with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
            texts.append(file.read())
    return texts

# Read Telugu text data from train directory
telugu_texts = read_telugu_text_files(train_data_path)

In [ ]:
# Generate summaries for each Telugu text in train data
for i, text in enumerate(telugu_texts):
    summary = generate_extractive_summary(text)
    print(f"Summary for text {i+1}:\n{summary}\n")


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Summary for text 1:
సబ్ పోస్టాఫీసు సౌకర్యం, పోస్ట్ అండ్ టెలిగ్రాఫ్ ఆఫీసు గ్రామానికి 5 కి.మీ. ఇంటర్నెట్ కెఫె / సామాన్య సేవా కేంద్రం, ప్రైవేటు కొరియర్ గ్రామం నుండి 10 కి.మీ.కి పైబడిన దూరంలో ఉన్నాయి. ప్రైవేటు బస్సు సౌకర్యం, రైల్వే స్టేషన్ మొదలైనవి గ్రామం నుండి 10 కి.మీ.కి పైబడిన దూరంలో ఉన్నాయి.

Summary for text 2:
కథకుడు వంతలు సంగీత వాద్యాలు పైన చెప్పిన శారదకథ వంటి కళారూపాలలో ఉన్నట్టు ఉన్నా పట ప్రదర్శన ఇక్కడి కళాప్రదర్శనలో ప్రత్యేకత. కళాప్రదర్శన పరిణామ క్రమంలో తోలు బొమ్మలాటకు ఈ పటం కథలను తొలి రూపాలుగా చెప్పడానికి వీలుంది. ప్రదర్శన కళలలో మానవేతర పాత్రధారులతో ఉన్న కళారూపాలు తెలుగు సంప్రదాయంలో ఎలా ఉన్నాయో చూశాము.

Summary for text 3:
పోస్టాఫీసు సౌకర్యం, పోస్ట్ అండ్ టెలిగ్రాఫ్ ఆఫీసు గ్రామం నుండి 10 కి.మీ.కి పైబడిన దూరంలో ఉన్నాయి. ఏటీఎమ్, సహకార బ్యాంకు గ్రామం నుండి 10 కి.మీ.కి పైబడిన దూరంలో ఉన్నాయి. రోజువారీ మార్కెట్, వ్యవసాయ మార్కెటింగ్ సొసైటీ గ్రామం నుండి 10 కి.మీ.కి పైబడిన దూరంలో ఉన్నాయి.

Summary for text 4:
మీమాంస తత్త్వవేత్తలు వేదాలను చాటటం పవిత్రమైనదని, వేదాలు లోపభూయిష్టాలు కావని, ధర్మ

In [ ]:
!pip install rouge

In [13]:
import nltk
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

# Download NLTK data if not already downloaded
nltk.download('punkt')

# Function to calculate BLEU score
def calculate_bleu(reference, summary):
    reference_tokens = nltk.word_tokenize(reference.lower())
    summary_tokens = nltk.word_tokenize(summary.lower())
    return sentence_bleu([reference_tokens], summary_tokens)

# Function to calculate ROUGE score
def calculate_rouge(reference, summary):
    rouge = Rouge()
    scores = rouge.get_scores(summary, reference)
    rouge_1_recall = scores[0]['rouge-1']['r']
    rouge_1_precision = scores[0]['rouge-1']['p']
    rouge_1_f1 = scores[0]['rouge-1']['f']
    return rouge_1_recall, rouge_1_precision, rouge_1_f1


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [14]:

# Inference function to generate summaries using Telugu BERT model
def generate_summary(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary

# Example Telugu text for evaluation
reference_text = "ఈ మండల కేంద్రమైన ముంచింగిపుట్టు నుండి 4 కి. మీ. దూరం లోను, సమీప పట్టణమైన జైపూరు నుండి 134 కి. మీ. దూరంలోనూ ఉంది."

# Generate summary using inference function
generated_summary = generate_summary(reference_text)

# Calculate BLEU score
bleu_score = calculate_bleu(reference_text, generated_summary)
print("BLEU Score:", bleu_score)

# Calculate ROUGE score
rouge_recall, rouge_precision, rouge_f1 = calculate_rouge(reference_text, generated_summary)
print("ROUGE-1 Recall:", rouge_recall)
print("ROUGE-1 Precision:", rouge_precision)
print("ROUGE-1 F1 Score:", rouge_f1)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


BLEU Score: 0.45579401832801714
ROUGE-1 Recall: 0.5625
ROUGE-1 Precision: 1.0
ROUGE-1 F1 Score: 0.7199999953920001


In [16]:

# Function to generate sentence embeddings using Telugu BERT
def generate_sentence_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        input_ids = tokenizer.encode(sentence, add_special_tokens=True, max_length=128, truncation=True, return_tensors="pt")
        with torch.no_grad():
            outputs = model(input_ids)
            sentence_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
            embeddings.append(sentence_embedding)
    return np.array(embeddings)

# Function to calculate similarity matrix
def calculate_similarity_matrix(embeddings):
    similarity_matrix = np.dot(embeddings, embeddings.T)
    return similarity_matrix


In [17]:

# Inference function to generate summaries using Telugu BERT model
def generate_summary_from_file(file_path, num_sentences=3):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary


In [20]:

# Example file path for inference
file_path = "/content/dataset/data/valid/208.txt"

# Generate summary using inference function
generated_summary = generate_summary_from_file(file_path)

print("Generated Summary:")
print(generated_summary)

Generated Summary:
1913లో, ఆమెకు తొమ్మిదేళ్ళ వయసులో ఉన్నప్పుడు, ఆమె తల్లి జయశ్రీబాయి జంబౌలింలోని ఒక గుడిలో ఉండే పూజారి దగ్గర సంగీతం నేర్చుకునేందుకు చేర్చింది. ఆ తరువాత మోగుబాయిని చంద్రేశ్వర్ భూతనాథ్ సంగీత మండలీ అనే ట్రావెలింగ్ నాటక కంపెనీలో చేర్చింది ఆమె తల్లి. ఆమె ఆ నాటక మండలిలో పనిచేసేటప్పుడు, 1914లో మోగు తల్లి మరణించింది.
